In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer

ROOT = Path.cwd().parent
print(ROOT)

DATA_RAW = ROOT/"data/raw"

INPUT_PATH = DATA_RAW / "00_model_df.csv"
OUTPUT_PATH = DATA_RAW / "01_embedded_model_df.csv"

model_df = scrape_df = pd.read_csv(INPUT_PATH)

c:\Users\sebas\PycharmProjects\Git\BoxOffice_Oracle


In [2]:
# Load small embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Fill missing text with empty string
texts = model_df["final_market_synopsis"].fillna("").astype(str).tolist()

# Create embeddings
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=False, #Need to download extra stuff for it
    normalize_embeddings=True
)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(2255, 384)

In [3]:
embedding_cols = [f"g7_market_synopsis_emb_{i}" for i in range(embeddings.shape[1])]

embeddings_df = pd.DataFrame(
    embeddings,
    columns=embedding_cols,
    index=model_df.index
)

In [4]:
embeddings_df.head()

,g7_market_synopsis_emb_0,g7_market_synopsis_emb_1,g7_market_synopsis_emb_2,g7_market_synopsis_emb_3,g7_market_synopsis_emb_4,g7_market_synopsis_emb_5,g7_market_synopsis_emb_6,g7_market_synopsis_emb_7,g7_market_synopsis_emb_8,g7_market_synopsis_emb_9,...,g7_market_synopsis_emb_374,g7_market_synopsis_emb_375,g7_market_synopsis_emb_376,g7_market_synopsis_emb_377,g7_market_synopsis_emb_378,g7_market_synopsis_emb_379,g7_market_synopsis_emb_380,g7_market_synopsis_emb_381,g7_market_synopsis_emb_382,g7_market_synopsis_emb_383
0,-0.047242,0.077914,-0.019149,0.079698,-0.013608,0.022451,0.060486,0.041957,0.069777,-0.014001,...,-0.010731,-0.014368,-0.078022,0.003610,-0.058226,-0.032532,-0.038527,-0.072058,-0.074371,-0.039796
1,-0.010995,0.069406,-0.078293,0.049609,0.023734,-0.025751,0.051487,-0.130534,0.025276,-0.026513,...,0.076011,-0.122633,0.034158,0.081395,0.050829,0.028579,0.025510,0.038926,0.038335,-0.021903
2,-0.107675,-0.021693,0.049794,-0.004289,0.011498,-0.003202,-0.065933,0.044007,0.057521,0.061340,...,-0.062428,0.013874,-0.101026,0.095624,0.053449,0.027899,0.081364,-0.019306,0.028493,-0.003193
3,-0.041832,0.062533,-0.151712,-0.007567,0.023233,0.058806,0.034648,-0.091888,-0.026586,-0.056178,...,0.031220,-0.089104,0.004114,-0.008470,0.000972,0.000932,0.062636,-0.084109,-0.054190,-0.075343
4,-0.017671,0.055290,0.021184,-0.109618,-0.016490,0.022886,0.063836,0.055152,-0.012061,0.045544,...,0.020103,-0.030943,-0.050717,0.072772,-0.006696,0.006582,-0.032968,-0.043960,0.058457,-0.035927


In [5]:
model_df_embedded = pd.concat(
    [model_df, embeddings_df],
    axis=1
)

model_df_embedded.shape

(2255, 426)

In [6]:
model_df_embedded.to_csv(
    DATA_RAW / "fe_groups/g7_market_synopsis_embeddings.csv",
    index=False
)